Импорт библиотек

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


Просмотр сырых данных

In [4]:
data_raw = pd.read_excel("../data/raw/Global Superstore.xls", parse_dates=['Order Date', 'Ship Date'])

data_raw.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,City,State,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority
0,32298,CA-2012-124891,2012-07-31,2012-07-31,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,...,TEC-AC-10003033,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...,2309.650,7,0.0,762.1845,933.57,Critical
1,26341,IN-2013-77878,2013-02-05,2013-02-07,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,...,FUR-CH-10003950,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",3709.395,9,0.1,-288.7650,923.63,Critical
2,25330,IN-2013-71249,2013-10-17,2013-10-18,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,...,TEC-PH-10004664,Technology,Phones,"Nokia Smart Phone, with Caller ID",5175.171,9,0.1,919.9710,915.49,Medium
3,13524,ES-2013-1579342,2013-01-28,2013-01-30,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,...,TEC-PH-10004583,Technology,Phones,"Motorola Smart Phone, Cordless",2892.510,5,0.1,-96.5400,910.16,Medium
4,47221,SG-2013-4320,2013-11-05,2013-11-06,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,...,TEC-SHA-10000501,Technology,Copiers,"Sharp Wireless Fax, High-Speed",2832.960,8,0.0,311.5200,903.04,Critical


# EDA

Просмотр типов данных и пропусков

In [5]:
data_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row ID          51290 non-null  int64         
 1   Order ID        51290 non-null  object        
 2   Order Date      51290 non-null  datetime64[ns]
 3   Ship Date       51290 non-null  datetime64[ns]
 4   Ship Mode       51290 non-null  object        
 5   Customer ID     51290 non-null  object        
 6   Customer Name   51290 non-null  object        
 7   Segment         51290 non-null  object        
 8   City            51290 non-null  object        
 9   State           51290 non-null  object        
 10  Country         51290 non-null  object        
 11  Postal Code     9994 non-null   float64       
 12  Market          51290 non-null  object        
 13  Region          51290 non-null  object        
 14  Product ID      51290 non-null  object        
 15  Ca

Пропуски в столбце Postal Code - это нормально, не у каждой страны есть Postal Code. Типы данных верные

Построим перекрестную таблицу по Country и Postal Code, чтобы узнать, по какой причине пропуски в столбце Postal Code

In [ ]:
crosstab_orders = pd.crosstab(
    data_raw["Postal Code"],
    data_raw["Country"],
    values=data_raw["Order ID"],
    aggfunc='count'
)

crosstab_orders

Country,United States
Postal Code,
1040.0,1
1453.0,6
1752.0,2
1810.0,4
1841.0,33
...,...
98502.0,5
98632.0,3
98661.0,5


In [23]:
data_raw["Country"].value_counts()

Country
United States    9994
Australia        2837
France           2827
Mexico           2644
Germany          2065
                 ... 
South Sudan         2
Bahrain             2
Swaziland           2
Burundi             2
Eritrea             2
Name: count, Length: 147, dtype: int64

Оказалось, что в датасете Postal Code указан лишь для заказов из США, хотя другие страны в датасете тоже есть.

Посмотрим статистики по числовым признакам

In [8]:
num_cols = ["Sales", "Quantity", "Discount", "Profit", "Shipping Cost"]

data_raw[num_cols].describe()

,Sales,Quantity,Discount,Profit,Shipping Cost
count,51290.000000,51290.000000,51290.000000,51290.000000,51290.000000
mean,246.490581,3.476545,0.142908,28.610982,26.375818
std,487.565361,2.278766,0.212280,174.340972,57.296810
min,0.444000,1.000000,0.000000,-6599.978000,0.002000
25%,30.758625,2.000000,0.000000,0.000000,2.610000
50%,85.053000,3.000000,0.000000,9.240000,7.790000
75%,251.053200,5.000000,0.200000,36.810000,24.450000
max,22638.480000,14.000000,0.850000,8399.976000,933.570000


Посмотрим временной период всех заказов и создадим на основе Order Date столбцы Order Year, Order Quarter, Order Month, Order Day, и аналогично для Ship Date

In [27]:
print(f"Период заказов: с {data_raw['Order Date'].min()} по {data_raw['Order Date'].max()}")

print(f"Период доставок: с {data_raw['Ship Date'].min()} по {data_raw['Ship Date'].max()}")

Период заказов: с 2011-01-01 00:00:00 по 2014-12-31 00:00:00
Период доставок: с 2011-01-03 00:00:00 по 2015-01-07 00:00:00


In [32]:
data_processed = data_raw.copy()

data_processed["Order Year"] = data_processed["Order Date"].dt.year
data_processed["Order Quarter"] = data_processed["Order Date"].dt.quarter
data_processed["Order Month"] = data_processed["Order Date"].dt.month
data_processed["Order Day"] = data_processed["Order Date"].dt.day

data_processed["Ship Year"] = data_processed["Ship Date"].dt.year
data_processed["Ship Quarter"] = data_processed["Ship Date"].dt.quarter
data_processed["Ship Month"] = data_processed["Ship Date"].dt.month
data_processed["Ship Day"] = data_processed["Ship Date"].dt.day

data_processed[["Order Date", "Order Year", "Order Quarter", "Order Month", "Order Day"]]

,Order Date,Order Year,Order Quarter,Order Month,Order Day
0,2012-07-31,2012,3,7,31
1,2013-02-05,2013,1,2,5
2,2013-10-17,2013,4,10,17
3,2013-01-28,2013,1,1,28
4,2013-11-05,2013,4,11,5
...,...,...,...,...,...
51285,2014-06-19,2014,2,6,19
51286,2014-06-20,2014,2,6,20
51287,2013-12-02,2013,4,12,2
51288,2012-02-18,2012,1,2,18


Создадим календарь для более удобной работы в Power BI

In [35]:
calendar_df = pd.DataFrame({
    'Date': pd.date_range(
        start=data_processed['Order Date'].min(),
        end=data_processed['Ship Date'].max(),
        freq='D'
    )
})

calendar_df['Year'] = calendar_df['Date'].dt.year
calendar_df['Month'] = calendar_df['Date'].dt.month
calendar_df['Month_Name'] = calendar_df['Date'].dt.strftime('%B')
calendar_df['Quarter'] = calendar_df['Date'].dt.quarter
calendar_df['Weekday'] = calendar_df['Date'].dt.day_name()
calendar_df['Week_Num'] = calendar_df['Date'].dt.isocalendar().week

calendar_df.to_csv('../data/processed/calendar.csv', index=False)

calendar_df

,Date,Year,Month,Month_Name,Quarter,Weekday,Week_Num
0,2011-01-01,2011,1,January,1,Saturday,52
1,2011-01-02,2011,1,January,1,Sunday,52
2,2011-01-03,2011,1,January,1,Monday,1
3,2011-01-04,2011,1,January,1,Tuesday,1
4,2011-01-05,2011,1,January,1,Wednesday,1
...,...,...,...,...,...,...,...
1463,2015-01-03,2015,1,January,1,Saturday,1
1464,2015-01-04,2015,1,January,1,Sunday,1
1465,2015-01-05,2015,1,January,1,Monday,2
1466,2015-01-06,2015,1,January,1,Tuesday,2
